# Milestone 1: airline policy graph

**What this is:** the airline policy from `wiki.md`, written as a graph in `airline_policy_graph.json`. Same rules as the prose, only the form is different. All 4 KiL-related systems (validator, shallow, semi-deep, deep) use this one graph.

**How it was made:** drafted by Claude from `wiki.md`, cross-checked by Muse, then verified by hand by me against `wiki.md` on 2026-10-26.

**GPU:** not needed. Use a CPU runtime.

In [ ]:
# mount Google Drive, where the graph file lives
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# check the graph: missing fields, unused wiki.md bullets, unverified items
%cd /content/drive/MyDrive/cmsc691-10-project2
!python check.py

In [ ]:
# quick look: each action and how many rules it has
import json
g = json.load(open("/content/drive/MyDrive/cmsc691-10-project2/airline_policy_graph.json"))

def count(conds):
    n = 0
    for c in conds:
        if "any_of" in c or "all_of" in c:
            n += count(c.get("any_of", []) + c.get("all_of", []))
        else:
            n += 1
    return n

print(f"{'action':32} {'info':>4} {'rules':>5}  sources")
print("-" * 70)
for a in g["actions"]:
    print(f"{a['action']:32} {len(a['needs_info']):>4} {count(a['conditions']):>5}  {', '.join(a['read'])}")

## Load the graph into Neo4j Aura (free)

**Graph model**
- `(:Action)` the 7 actions, plus `transfer_to_human_agents`
- `(:Info)` what the agent must ask first; `(:Action)-[:NEEDS_INFO {order, source}]->(:Info)`
- `(:Condition)` one rule; has `text, actor, check, source, if_fails`
- `(:Gate {kind: OR/AND})` groups conditions; `(:Gate)-[:BRANCH]->(...)`
- `(:Action)-[:REQUIRES]->(:Condition or :Gate)`
- `(:GlobalRule)` G1–G5; G1 has `APPLIES_TO` edges to its 5 actions
- `(:Fact)`, `(:Table)`, `(:Description)` notes, the bag table, and D1/D2/C4

Every node also has the label `:PolicyGraph`, so it lives safely next to other data (e.g. Project 1) in the same free database.

The JSON stays the master copy. If a rule changes, edit the JSON and load again.

**Before running:** put the 3 values from the Aura txt file into Colab Secrets (key icon on the left): `NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD`. Turn on notebook access for all 3. Free Aura pauses after a few days without use; click **Resume** in the Aura console first.

In [ ]:
# install the Neo4j Python driver and pyvis (to draw the graph)
!pip install -q neo4j pyvis

In [ ]:
# get the Aura login from Colab Secrets
import os
from google.colab import userdata
for k in ["NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD"]:
    os.environ[k] = userdata.get(k)
print("loaded:", os.environ["NEO4J_URI"])

In [ ]:
# load the JSON into Neo4j
# it deletes only the old policy graph (nodes labeled :PolicyGraph), then loads it again
# other data in the database (e.g. Project 1) is not touched
%cd /content/drive/MyDrive/cmsc691-10-project2
!python load_neo4j.py

In [ ]:
# check: Neo4j has the same number of rules per action as the JSON
import json
from neo4j import GraphDatabase

g = json.load(open("/content/drive/MyDrive/cmsc691-10-project2/airline_policy_graph.json"))
def count(conds):
    return sum(count(c.get("any_of", []) + c.get("all_of", [])) if ("any_of" in c or "all_of" in c) else 1
               for c in conds)

drv = GraphDatabase.driver(os.environ["NEO4J_URI"],
                           auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"]))
q = '''
MATCH (a:Action:PolicyGraph {name:$name})
OPTIONAL MATCH (a)-[:REQUIRES|BRANCH*1..5]->(c:Condition)
WITH a, count(DISTINCT c) AS rules
OPTIONAL MATCH (a)-[:NEEDS_INFO]->(i:Info)
RETURN rules, count(i) AS info
'''
ok = True
with drv.session() as s:
    for a in g["actions"]:
        r = s.run(q, name=a["action"]).single()
        want = (count(a["conditions"]), len(a["needs_info"]))
        got = (r["rules"], r["info"])
        flag = "✅" if got == want else "❌"
        ok &= got == want
        print(f"{flag} {a['action']:32} rules {got[0]}/{want[0]}  info {got[1]}/{want[1]}")
print("\nNeo4j matches the JSON" if ok else "\nMISMATCH: check the loader")

In [ ]:
# example query: what must be true to cancel a reservation?
q = '''
MATCH path = (a:Action:PolicyGraph {name:'cancel_reservation'})-[:REQUIRES|BRANCH*1..5]->(c:Condition)
RETURN [n IN nodes(path)[1..-1] | n.kind] AS gates, c.text AS rule, c.check AS check, c.source AS src
'''
with drv.session() as s:
    for r in s.run(q):
        gates = " > ".join(g for g in r["gates"] if g) or "-"
        print(f"{r['src']:3} {r['check']:9} {gates:9} {r['rule']}")

## See the graph

Option 1: the cell below draws one action. Change `ACTION` to see another one.

Option 2: in the Aura console, open **Query** and run:

```
MATCH p = (a:Action:PolicyGraph)-[*1..3]->(n:PolicyGraph) RETURN p
```

Take screenshots for the report.

In [ ]:
# draw one action from Neo4j as an interactive picture
ACTION = "cancel_reservation"
from pyvis.network import Network
from IPython.display import HTML, display

COLORS = {"Action": "#dae8fc", "Condition": "#d5e8d4", "Gate": "#fff2cc", "Info": "#eeeeee",
          "Fact": "#f8cecc", "Table": "#e1d5e7", "GlobalRule": "#ffe6cc"}
q = '''
MATCH p = (a:Action:PolicyGraph {name:$name})-[*1..4]->(n:PolicyGraph) RETURN p
UNION
MATCH p = (r:GlobalRule:PolicyGraph)-[:APPLIES_TO]->(a:Action:PolicyGraph {name:$name}) RETURN p
'''
net = Network(height="650px", width="100%", directed=True, cdn_resources="in_line")
with drv.session() as s:
    for rec in s.run(q, name=ACTION):
        path = rec["p"]
        for n in path.nodes:
            label = [l for l in n.labels if l != "PolicyGraph"][0]
            text = n.get("name") or n.get("kind") or n.get("text") or n.get("source")
            if n.get("check") == "dialogue":
                color = "#ffe6cc"
            else:
                color = COLORS.get(label, "#ffffff")
            src = f" ({n.get('source')})" if n.get("source") and label != "GlobalRule" else ""
            net.add_node(n.element_id, label=(text[:45] + src), title=f"{label}: {text}", color=color,
                         shape="box")
        for r in path.relationships:
            net.add_edge(r.start_node.element_id, r.end_node.element_id, label=r.type)
display(HTML(net.generate_html()))

## Next
- Build the runtime validator from this graph (it checks the `check = data` rules before each tool call).